## Analisis de lenguaje en reviews de vino



Este proyecto estudia como cambian las palabras usadas para describir distintas cepas de vino. La lectura se centra en aromas y sabores, no en calidad ni puntaje, porque primero se necesita entender el vocabulario base de cada variedad.

El resultado cumple dos funciones: servir como pieza de portafolio y preparar los datos para una futura app o pagina web donde sea facil navegar cepas, palabras, graficos radiales y mapas locales.

La primera celda carga librerias, define rutas y fija una semilla para que los resultados se puedan repetir.


In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

DATA_DIR = Path.cwd()
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR

## Limpieza de texto y clasificacion de vinos



Esta etapa normaliza texto, corrige acentos y agrupa nombres que representan la misma idea. Por ejemplo, Muscat y Moscato se tratan como la misma cepa cuando el contexto lo permite.

Tambien se asigna cada vino a un grupo amplio, como tinto, blanco, espumante o rosado. Esa separacion evita comparar palabras de tintos contra blancos cuando el analisis requiere contextos distintos.

Las etiquetas genericas, como Italian Red o French White, se reducen a la cepa real cuando aparece una variedad mas informativa.


In [ ]:
def clean_text_series(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
        .str.replace("\u00a0", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )


def strip_accents(value: str) -> str:
    if pd.isna(value):
        return ""
    value = str(value)
    if "\u00c3" in value or "\u00c2" in value:
        try:
            value = value.encode("latin1").decode("utf-8")
        except UnicodeError:
            pass
    value = unicodedata.normalize("NFKD", value)
    return "".join(ch for ch in value if not unicodedata.combining(ch))


RED_HINTS = {
    "aglianico", "barbera", "bonarda", "bordeaux-style red blend", "cabernet", "cabernet franc",
    "cabernet sauvignon", "carignan", "carmenere", "corvina", "gamay", "grenache", "malbec",
    "meritage", "merlot", "mourvedre", "nebbiolo", "nero d'avola", "petite sirah", "petit verdot",
    "pinot noir", "portuguese red", "red blend", "rhone-style red blend", "sangiovese", "syrah",
    "shiraz", "tempranillo", "tinta de toro", "touriga nacional", "zinfandel"
}

WHITE_HINTS = {
    "albarino", "chenin blanc", "chardonnay", "gewurztraminer", "gruner veltliner", "moscato",
    "pinot blanc", "pinot grigio", "pinot gris", "portuguese white", "riesling", "sauvignon",
    "sauvignon blanc", "semillon", "torrontes", "verdejo", "vermentino", "viognier", "white blend"
}

SPARKLING_HINTS = {
    "sparkling", "sparkling blend", "champagne", "prosecco", "cava", "franciacorta", "cremant", "sekt"
}


def infer_color(category, variety) -> str:
    category_norm = strip_accents(category).lower().strip()
    variety_norm = strip_accents(variety).lower().strip()

    if category_norm in {"red", "white", "rose", "sparkling", "dessert"}:
        if category_norm == "rose":
            return "rose"
        return category_norm

    for hint in SPARKLING_HINTS:
        if hint in variety_norm:
            return "sparkling"

    for hint in RED_HINTS:
        if hint in variety_norm:
            return "red"
    for hint in WHITE_HINTS:
        if hint in variety_norm:
            return "white"
    return "unknown"


CANONICAL_VARIETY_TERMS = [
    ("cabernet sauvignon", "Cabernet Sauvignon"),
    ("cabernet franc", "Cabernet Franc"),
    ("sauvignon blanc", "Sauvignon Blanc"),
    ("pinot noir", "Pinot Noir"),
    ("pinot gris", "Pinot Gris"),
    ("pinot grigio", "Pinot Grigio"),
    ("pinot blanc", "Pinot Blanc"),
    ("chenin blanc", "Chenin Blanc"),
    ("petite sirah", "Petite Sirah"),
    ("petit verdot", "Petit Verdot"),
    ("nero d avola", "Nero d'Avola"),
    ("bordeaux style red", "Bordeaux-style Red Blend"),
    ("bordeaux style white", "Bordeaux-style White Blend"),
    ("rhone style red", "Rhone-style Red Blend"),
    ("rhone style white", "Rhone-style White Blend"),
    ("red blend", "Blend"),
    ("white blend", "Blend"),
    ("sparkling blend", "Blend"),
    ("meritage", "Blend"),
    ("portuguese red", "Portuguese Red"),
    ("portuguese white", "Portuguese White"),
    ("muscat moscato", "Moscato"),
    ("muscat blanc", "Moscato"),
    ("muscat canelli", "Moscato"),
    ("moscato", "Moscato"),
    ("muscat", "Moscato"),
    ("chardonnay", "Chardonnay"),
    ("carmenere", "Carmenere"),
    ("dolcetto", "Dolcetto"),
    ("primitivo", "Primitivo"),
    ("montepulciano", "Montepulciano"),
    ("carignan", "Carignan"),
    ("grenache", "Grenache"),
    ("garnacha", "Grenache"),
    ("mourvedre", "Mourvedre"),
    ("malbec", "Malbec"),
    ("merlot", "Merlot"),
    ("syrah", "Syrah"),
    ("shiraz", "Syrah"),
    ("riesling", "Riesling"),
    ("tempranillo", "Tempranillo"),
    ("sangiovese", "Sangiovese"),
    ("zinfandel", "Zinfandel"),
    ("nebbiolo", "Nebbiolo"),
    ("barbera", "Barbera"),
    ("gamay", "Gamay"),
    ("albarino", "Albarino"),
    ("viognier", "Viognier"),
    ("verdejo", "Verdejo"),
    ("torrontes", "Torrontes"),
    ("gewurztraminer", "Gewurztraminer"),
    ("gruner veltliner", "Gruner Veltliner"),
]

GENERIC_VARIETY_PATTERNS = [
    r"\bitalian\s+(red|white)\b", r"\bfrench\s+(red|white)\b", r"\bspanish\s+(red|white)\b",
    r"\bcalifornia\s+(red|white)\b", r"\bproprietary\s+(red|white)\b",
    r"\bsweet\s+(red|white)\b", r"\bdessert\s+(red|white)\b",
]


def normalize_variety_text(value: str) -> str:
    text = strip_accents(value).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def title_case_variety(value: str) -> str:
    text = normalize_variety_text(value)
    return " ".join(part.capitalize() for part in text.split()) if text else "Unknown"


def extract_variety_components(value: str) -> list[str]:
    norm = normalize_variety_text(value)
    if not norm:
        return []
    generic_norm = norm
    for pattern in GENERIC_VARIETY_PATTERNS:
        generic_norm = re.sub(pattern, " ", generic_norm)
    generic_norm = re.sub(r"\s+", " ", generic_norm).strip()
    if not generic_norm:
        return []
    components = []
    for needle, canonical in CANONICAL_VARIETY_TERMS:
        if re.search(rf"\b{re.escape(needle)}\b", generic_norm):
            components.append(canonical)
    components = list(dict.fromkeys(components))
    if re.search(r"\bcabernet\b", generic_norm) and "Cabernet Sauvignon" not in components and "Cabernet Franc" not in components:
        components.append("Cabernet Sauvignon")
    components = list(dict.fromkeys(components))
    is_blend = "blend" in norm or len(components) > 1
    if is_blend:
        components.append("Blend")
    if not components:
        components.append(title_case_variety(generic_norm))
    return list(dict.fromkeys(components))

## Carga de los datasets



Los tres CSV se unen en una tabla comun con columnas equivalentes. Esta tabla contiene la fuente, variedad, categoria, review, puntaje, precio, bodega, titulo y lugar.

Los blends se expanden cuando contienen cepas reconocibles. Un Cabernet-Syrah suma evidencia para Cabernet, Syrah y Blend, porque la review describe un vino que pertenece a mas de una familia.

La tabla resultante permite analizar todas las fuentes con una sola logica.


In [ ]:
def load_sources(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    frames = []

    wine_path = data_dir / "wine.csv"
    if wine_path.exists():
        wine = pd.read_csv(wine_path)
        frame = pd.DataFrame({
            "source": "wine.csv",
            "variety": wine["varietal"],
            "category": wine["category"],
            "review_text": wine["review"],
            "score": pd.to_numeric(wine["rating"], errors="coerce"),
            "price": pd.to_numeric(wine["price"].astype("string").str.replace("$", "", regex=False), errors="coerce"),
            "winery": wine["winery"],
            "title": wine["wine"],
            "place": wine["appellation"],
        })
        frames.append(frame)

    wm130_path = data_dir / "winemag-data-130k-v2.csv"
    if wm130_path.exists():
        wm130 = pd.read_csv(wm130_path, index_col=0)
        frame = pd.DataFrame({
            "source": "winemag-data-130k-v2.csv",
            "variety": wm130["variety"],
            "category": pd.NA,
            "review_text": wm130["description"],
            "score": pd.to_numeric(wm130["points"], errors="coerce"),
            "price": pd.to_numeric(wm130["price"], errors="coerce"),
            "winery": wm130["winery"],
            "title": wm130["title"],
            "place": wm130["country"].fillna("") + " | " + wm130["province"].fillna("") + " | " + wm130["region_1"].fillna(""),
        })
        frames.append(frame)

    wm150_path = data_dir / "winemag-data_first150k.csv"
    if wm150_path.exists():
        wm150 = pd.read_csv(wm150_path, index_col=0)
        frame = pd.DataFrame({
            "source": "winemag-data_first150k.csv",
            "variety": wm150["variety"],
            "category": pd.NA,
            "review_text": wm150["description"],
            "score": pd.to_numeric(wm150["points"], errors="coerce"),
            "price": pd.to_numeric(wm150["price"], errors="coerce"),
            "winery": wm150["winery"],
            "title": pd.NA,
            "place": wm150["country"].fillna("") + " | " + wm150["province"].fillna("") + " | " + wm150["region_1"].fillna(""),
        })
        frames.append(frame)

    df = pd.concat(frames, ignore_index=True)
    for col in ["variety", "category", "review_text", "winery", "title", "place"]:
        df[col] = clean_text_series(df[col])
    df = df.dropna(subset=["variety", "review_text"]).reset_index(drop=True)
    df["variety_raw"] = df["variety"]
    df["variety_components"] = df["variety_raw"].map(extract_variety_components)
    df["is_blend_source"] = df["variety_components"].map(lambda values: "Blend" in values or len(values) > 1)
    df = df.explode("variety_components").dropna(subset=["variety_components"]).reset_index(drop=True)
    df["variety"] = df["variety_components"].astype("string")
    df["color_group"] = [
        infer_color(cat, raw if var == "Blend" else var)
        for cat, raw, var in zip(df["category"], df["variety_raw"], df["variety"])
    ]
    df = df.drop(columns=["variety_components"])
    return df


analysis_df = load_sources()
analysis_df.shape

## Auditoria inicial de la carga



Estas tablas muestran algunas filas y cuentan registros por fuente, color y variedad. Funcionan como control de calidad antes de construir graficos.

Para un lector no tecnico, esta parte responde si el dataset se cargo con volumen suficiente y si las categorias principales tienen sentido visualmente.

Una mala carga en esta etapa contaminaria todos los graficos posteriores.


In [ ]:
display(analysis_df.head(3))
display(analysis_df["source"].value_counts())
display(analysis_df["color_group"].value_counts(dropna=False))
display(analysis_df["variety"].value_counts().head(25))
display(
    analysis_df.loc[analysis_df["is_blend_source"], ["variety_raw", "variety", "color_group"]]
    .drop_duplicates()
    .head(25)
)

## Diccionario de descriptores



Las reviews se convierten en un diccionario de palabras utiles para describir aromas y sabores. Se eliminan conectores, nombres de cepa, palabras narrativas y terminos de estructura que no distinguen una variedad.

Los plurales y sinonimos se unifican: cherries queda como cherry, minerality queda como mineral y cacao queda como cocoa. Esta normalizacion evita dividir una misma idea en varias columnas pequenas.

Los grupos semanticos conectan conceptos relacionados sin inventar notas especificas. Citrus suma al grupo citrus, pero una nota como yuzu solo crece si aparece por si misma con suficiente presencia.


In [ ]:
ENGLISH_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but", "by", "for", "from", "has", "have",
    "in", "into", "is", "it", "its", "of", "on", "or", "that", "the", "this", "to", "was", "were",
    "with", "while", "will", "would", "you", "your", "their", "there", "these", "those", "than", "then",
    "so", "some", "such", "also", "more", "most", "less", "very", "quite", "rather", "much", "many",
    "now", "just", "still", "can", "could", "should", "may", "might", "not", "no", "nor", "all", "any",
    "he", "she", "they", "we", "i", "s", "t", "ll", "ve", "re", "d", "m",
    "because", "about", "after", "before", "over", "under", "up", "down", "out", "off", "too",
    "when", "where", "what", "which", "who", "whom", "whose", "why", "how", "only", "even", "ever", "yet",
    "again", "already", "near", "nearly", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten"
}

REVIEW_STRUCTURE_STOPWORDS = {
    "wine", "wines", "drink", "drinking", "bottle", "bottling", "palate", "finish", "finishes",
    "flavor", "flavors", "aroma", "aromas", "note", "notes", "nose", "bouquet", "scent", "scents",
    "show", "shows", "showing", "shown", "offer", "offers", "offering", "offered", "open", "opens", "opening", "opened",
    "reveal", "reveals", "revealing", "revealed", "bring", "brings", "brought", "carry", "carries", "carrying", "carried",
    "lead", "leads", "leading", "suggest", "suggests", "suggesting", "made", "makes", "make", "give", "gives", "giving",
    "vineyard", "vineyards", "vines", "grape", "grapes", "varietal", "blend", "blends", "cuvee",
    "vintage", "year", "years", "old", "young", "age", "aged", "aging", "cellar", "cellaring",
    "winery", "producer", "production", "estate", "reserve", "selection", "county", "valley", "region", "cab", "cabs"
}

STRUCTURE_AND_QUALITY_STOPWORDS = {
    "acidity", "acid", "acidic", "bright", "crisp", "zesty", "brisk", "body", "bodied", "mouth",
    "mouthfeel", "texture", "textured", "structured", "structure", "tannin", "tannins", "tannic",
    "dry", "sweet", "sweetness", "semidry", "offdry", "fresh", "ripe", "rich", "soft", "firm", "full",
    "light", "medium", "good", "great", "fine", "nice", "simple", "clean", "balanced", "concentrated",
    "intense", "elegant", "complex", "easy", "approachable", "drinkable", "long", "short", "smooth",
    "well", "better", "best", "tasty", "delicious", "pleasant", "attractive"
}

GENERIC_UNIGRAMS = {
    "fruit", "fruits", "berry", "berries", "character", "style", "feel", "touch", "hint", "edge",
    "core", "sense", "tones", "elements", "alongside", "through", "throughout", "bit", "lot", "lots",
    "kind", "sort", "almost", "slightly", "mostly", "background", "front", "back", "end", "entry"
}

COLOR_MODIFIERS = {"black", "red", "white", "green", "yellow", "golden", "dark", "purple", "pink"}
TOKEN_RE = re.compile(r"[a-z]+")


def normalize_term(term: str) -> str:
    return strip_accents(term).lower().strip().replace(" ", "_").replace("-", "_")

TERM_CANONICAL = {
    "cherries": "cherry", "cherried": "cherry",
    "berries": "berry", "berryish": "berry",
    "raspberries": "raspberry", "strawberries": "strawberry", "cranberries": "cranberry",
    "currants": "currant", "blackcurrants": "blackcurrant", "blackberries": "blackberry",
    "plums": "plum", "figs": "fig", "apples": "apple", "pears": "pear", "peaches": "peach",
    "lemons": "lemon", "limes": "lime", "oranges": "orange", "grapefruits": "grapefruit",
    "pomelos": "pomelo", "bergamots": "bergamot", "citrons": "citron", "kumquats": "kumquat",
    "citrics": "citrus", "citric": "citrus", "citrusy": "citrus",
    "minerality": "mineral", "minerally": "mineral", "minerals": "mineral",
    "oaky": "oak", "oaked": "oak", "woody": "wood", "woods": "wood",
    "cacao": "cocoa", "cocoas": "cocoa", "chocolaty": "chocolate", "chocolatey": "chocolate", "chocolates": "chocolate",
    "spices": "spice", "spicy": "spice", "peppery": "pepper", "peppercorns": "pepper",
    "floral": "flower", "flowers": "flower", "violets": "violet",
}

RED_BERRY_TERMS = {"cherry", "raspberry", "strawberry", "cranberry", "currant"}
BLACK_FRUIT_TERMS = {"blackberry", "blackcurrant", "black_cherry", "plum", "cassis"}
CITRUS_TERMS = {"lemon", "lime", "grapefruit", "orange", "pomelo", "yuzu", "tangerine", "mandarin", "clementine", "bergamot", "buddha_hand", "citron", "kumquat"}
WOOD_TERMS = {"oak", "cedar", "cherry_wood", "sandalwood"}
SEMANTIC_GROUP_CHILDREN = {
    "citrus": CITRUS_TERMS,
    "red_berry": RED_BERRY_TERMS,
    "black_fruit": BLACK_FRUIT_TERMS,
    "wood": WOOD_TERMS,
}
GROUP_CHILD_MIN_REVIEW_SHARE = 0.05
GROUP_CHILD_BONUS_SHARE = 0.50
ORCHARD_TROPICAL_TERMS = {"apple", "pear", "peach", "apricot", "melon", "pineapple", "mango", "guava", "lychee"}
DESCRIPTOR_HEAD_TERMS = RED_BERRY_TERMS | BLACK_FRUIT_TERMS | CITRUS_TERMS | WOOD_TERMS | ORCHARD_TROPICAL_TERMS | {
    "berry", "fruit", "fig", "raisin", "prune", "spice", "pepper", "cocoa", "chocolate", "coffee", "vanilla", "toast", "smoke", "mineral", "flower", "violet"
}
PHRASE_MODIFIERS = COLOR_MODIFIERS | {"dried", "baked", "candied", "sour", "tart", "dark", "roasted", "toasted", "smoked", "bitter"}
PHRASE_TAILS = DESCRIPTOR_HEAD_TERMS | {"skin", "peel", "jam", "compote"}


def canonical_token(token: str) -> str:
    token = TERM_CANONICAL.get(token, token)
    if token.endswith("ies") and len(token) > 4:
        token = token[:-3] + "y"
    elif token.endswith("s") and len(token) > 4 and token not in {"cassis", "citrus"} and not token.endswith(("ss", "us", "ous", "is")):
        token = token[:-1]
    return TERM_CANONICAL.get(token, token)


def canonical_term(term: str) -> str:
    term = normalize_term(term)
    if "_" not in term:
        return canonical_token(term)
    parts = [canonical_token(part) for part in term.split("_") if part]
    return normalize_term("_".join(parts))


def expand_semantic_term(term: str) -> list[str]:
    term = canonical_term(term)
    expanded = [term]
    if term in CITRUS_TERMS:
        expanded.append("citrus")
    if term in WOOD_TERMS:
        expanded.append("wood")
    if term == "oak_wood":
        expanded.extend(["oak", "wood"])
    if term == "cherry_wood":
        expanded.append("wood")
    if term in RED_BERRY_TERMS:
        expanded.append("red_berry")
    if term in BLACK_FRUIT_TERMS:
        expanded.append("black_fruit")
    if "_" in term:
        parts = term.split("_")
        if parts[-1] in RED_BERRY_TERMS | CITRUS_TERMS | WOOD_TERMS | ORCHARD_TROPICAL_TERMS | {"cherry", "plum", "fig", "pepper", "spice", "cocoa", "chocolate", "coffee", "vanilla", "mineral", "flower", "violet"}:
            expanded.append(parts[-1])
    return list(dict.fromkeys(expanded))


def phrase_terms(value: str) -> set[str]:
    tokens = [canonical_token(token) for token in TOKEN_RE.findall(strip_accents(value).lower())]
    terms = set(tokens)
    terms.update(f"{left}_{right}" for left, right in zip(tokens, tokens[1:]))
    if len(tokens) > 1:
        terms.add("_".join(tokens))
    return {term for term in terms if len(term) >= 3}


VARIETY_STOP_TERMS = set()
variety_names_for_stopwords = pd.concat([analysis_df["variety"], analysis_df["variety_raw"]]).dropna().unique()
for variety_name in variety_names_for_stopwords:
    VARIETY_STOP_TERMS.update(phrase_terms(variety_name))
DESCRIPTOR_STOPWORD_EXCEPTIONS = DESCRIPTOR_HEAD_TERMS | PHRASE_MODIFIERS | PHRASE_TAILS | {
    "orange", "rose", "petal", "tea", "honey", "honeycomb", "almond", "hazelnut", "walnut"
}
VARIETY_STOP_TERMS = {term for term in VARIETY_STOP_TERMS if term not in DESCRIPTOR_STOPWORD_EXCEPTIONS}

GLOBAL_STOP_TERMS = (
    ENGLISH_STOPWORDS
    | REVIEW_STRUCTURE_STOPWORDS
    | STRUCTURE_AND_QUALITY_STOPWORDS
    | GENERIC_UNIGRAMS
    | VARIETY_STOP_TERMS
)
HARD_STOP_TERMS = ENGLISH_STOPWORDS | REVIEW_STRUCTURE_STOPWORDS | STRUCTURE_AND_QUALITY_STOPWORDS | VARIETY_STOP_TERMS


def review_tokens(text: str, variety=None) -> list[str]:
    if pd.isna(text):
        return []
    text = strip_accents(str(text)).lower()
    raw_tokens = TOKEN_RE.findall(text)
    own_variety_terms = phrase_terms(variety) if variety is not None else set()

    base_tokens = []
    for token in raw_tokens:
        token = canonical_token(token)
        if len(token) < 3:
            continue
        if token in HARD_STOP_TERMS or token in own_variety_terms:
            continue
        base_tokens.append(token)

    unigrams = [token for token in base_tokens if token not in COLOR_MODIFIERS]

    bigrams = []
    for left, right in zip(base_tokens, base_tokens[1:]):
        left = canonical_token(left)
        right = canonical_token(right)
        phrase = f"{left}_{right}"
        if phrase in GLOBAL_STOP_TERMS or phrase in own_variety_terms:
            continue
        if right in COLOR_MODIFIERS:
            continue
        if left == "buddha" and right == "hand":
            bigrams.append("buddha_hand")
            continue
        keep_phrase = (
            (left in PHRASE_MODIFIERS and right in PHRASE_TAILS)
            or (left in DESCRIPTOR_HEAD_TERMS and right in {"wood", "spice", "pepper", "skin", "peel", "jam", "compote"})
        )
        if keep_phrase:
            bigrams.append(phrase)

    expanded_terms = []
    for term in unigrams + bigrams:
        if term in GENERIC_UNIGRAMS or term in COLOR_MODIFIERS:
            continue
        expanded_terms.extend(expand_semantic_term(term))

    # Cap por resena: evita que una resena que repite 10 veces oak domine, pero conserva 2-3 menciones reales.
    capped = []
    for term, count in Counter(expanded_terms).items():
        capped.extend([term] * min(count, 3))
    return capped


def text_for_embedding(text: str, variety=None) -> str:
    return " ".join(review_tokens(text, variety=variety))

## Prueba visible de tokenizacion



Esta celda toma una review real y muestra las palabras que sobreviven al proceso de limpieza. La revision manual ayuda a detectar ruido antes de confiar en miles de conteos.

Si aparecen palabras como nombres de cepa, conectores o frases sin valor sensorial, el diccionario necesita ajustes. Si predominan frutas, especias, madera, flores o minerales, el filtro esta cumpliendo su funcion.


In [ ]:
sample_row = analysis_df.loc[0]
sample_text = sample_row["review_text"]
print(sample_row["variety"])
print(sample_text)
print(review_tokens(sample_text, variety=sample_row["variety"])[:40])

## Contadores por cepa



Cada cepa recibe un contador de palabras y un contador de presencia por review. El primer contador mide volumen; el segundo mide cuantas reviews distintas mencionan la palabra.

La presencia por review evita que una sola review que repite muchas veces oak domine el resultado. Se conserva algo de repeticion real, pero se limita su efecto.

La expansion condicionada reparte parte del peso de un grupo hacia una nota hija solo cuando esa nota ya aparece con fuerza minima suficiente.


In [ ]:
MIN_REVIEWS_PER_VARIETY = 200


def apply_conditional_group_expansion(counter: Counter, presence_counter: Counter, review_count: int):
    for group, children in SEMANTIC_GROUP_CHILDREN.items():
        group_count = counter.get(group, 0)
        group_hits = presence_counter.get(group, 0)
        if group_count <= 0 or group_hits <= 0:
            continue
        min_child_hits = max(1, math.ceil(group_hits * GROUP_CHILD_MIN_REVIEW_SHARE))
        count_bonus = max(1, int(round(group_count * GROUP_CHILD_BONUS_SHARE)))
        hit_bonus = max(1, int(round(group_hits * GROUP_CHILD_BONUS_SHARE)))
        for child in sorted(children):
            if child == group:
                continue
            if presence_counter.get(child, 0) >= min_child_hits:
                counter[child] += count_bonus
                presence_counter[child] = min(review_count, presence_counter.get(child, 0) + hit_bonus)
    return counter, presence_counter


def build_variety_counters(df: pd.DataFrame, min_reviews: int = MIN_REVIEWS_PER_VARIETY):
    counters = {}
    review_presence_counters = {}
    meta = []
    total_counter = Counter()

    grouped = df.groupby("variety", sort=True)
    for variety, group in grouped:
        review_count = len(group)
        if review_count < min_reviews:
            continue

        counter = Counter()
        presence_counter = Counter()
        for text in group["review_text"].dropna():
            tokens = review_tokens(text, variety=variety)
            counter.update(tokens)
            presence_counter.update(set(tokens))

        counter, presence_counter = apply_conditional_group_expansion(counter, presence_counter, review_count)

        if not counter:
            continue

        counters[variety] = counter
        review_presence_counters[variety] = presence_counter
        total_counter.update(counter)
        color = group["color_group"].mode(dropna=True)
        meta.append({
            "variety": variety,
            "review_count": review_count,
            "color_group": color.iloc[0] if len(color) else "unknown",
            "token_count": sum(counter.values()),
        })

    meta_df = pd.DataFrame(meta).sort_values("review_count", ascending=False).reset_index(drop=True)
    return counters, review_presence_counters, total_counter, meta_df


variety_counters, review_presence_counters, total_counter, variety_meta = build_variety_counters(analysis_df)
display(variety_meta.head(30))
len(variety_counters), len(total_counter)

## Buscador de cepas disponibles



Esta herramienta permite buscar nombres de cepas dentro del dataset ya normalizado. Es una ayuda practica para evitar errores de escritura antes de graficar.

La tabla resultante muestra cantidad de reviews, color asignado y volumen de tokens. Una cepa con pocas reviews puede ser menos confiable para comparaciones finas.


In [ ]:
def find_varieties(query: str, n: int = 20) -> pd.DataFrame:
    q = strip_accents(query).lower()
    mask = variety_meta["variety"].map(lambda x: q in strip_accents(x).lower())
    return variety_meta.loc[mask].head(n)


display(find_varieties("cabernet"))
display(find_varieties("carmen"))
display(find_varieties("chardonnay"))

## Ejercicio 1: graficos de barras



Los graficos de barras comparan palabras frecuentes dentro de una cepa o muestran en que cepas aparece mas una palabra buscada. Una barra mas larga significa mayor presencia bajo la metrica seleccionada.

Hay tres lecturas: apariciones totales, apariciones por 1000 tokens y porcentaje de reviews que mencionan el termino. El porcentaje suele ser mas justo porque no premia automaticamente a las cepas con mas rese?as.

Los filtros por tinto, blanco, espumante y rosado permiten comparar vinos dentro de familias mas parecidas.


In [ ]:
def normalize_term(term: str) -> str:
    return strip_accents(term).lower().strip().replace(" ", "_").replace("-", "_")


TERM_USAGE_MODES = {
    "count": ("count", "Apariciones totales"),
    "per_1000_tokens": ("per_1000_tokens", "Apariciones por 1000 tokens"),
    "review_pct": ("review_pct", "% de resenas que mencionan el termino"),
}


def top_terms_table(variety: str, n: int = 20, mode: str = "per_1000_tokens") -> pd.DataFrame:
    if variety not in variety_counters:
        raise ValueError(f"Variety not found or below min_reviews: {variety}")
    counter = variety_counters[variety]
    total = sum(counter.values())
    review_count = int(variety_meta.set_index("variety").loc[variety, "review_count"])
    metric, _ = TERM_USAGE_MODES.get(mode, TERM_USAGE_MODES["per_1000_tokens"])
    rows = []
    for term, count in counter.items():
        review_hits = review_presence_counters.get(variety, Counter()).get(term, 0)
        rows.append({
            "variety": variety,
            "term": term,
            "count": count,
            "per_1000_tokens": 1000 * count / total,
            "review_hits": int(review_hits),
            "review_pct": float(100 * review_hits / review_count) if review_count else 0.0,
        })
    return pd.DataFrame(rows).sort_values(metric, ascending=False).head(n).reset_index(drop=True)


def plot_top_terms(varieties: list[str], n: int = 20, mode: str = "per_1000_tokens"):
    metric, label = TERM_USAGE_MODES.get(mode, TERM_USAGE_MODES["per_1000_tokens"])
    fig, axes = plt.subplots(len(varieties), 1, figsize=(10, 4 * len(varieties)), constrained_layout=True)
    if len(varieties) == 1:
        axes = [axes]

    for ax, variety in zip(axes, varieties):
        table = top_terms_table(variety, n=n, mode=mode).sort_values(metric)
        ax.barh(table["term"], table[metric])
        ax.set_title(variety)
        ax.set_xlabel(label)
        ax.set_ylabel("")

    plt.show()



def normalize_color_filter(color_groups):
    if color_groups is None:
        return None
    if isinstance(color_groups, str):
        color_groups = [color_groups]
    expanded = []
    for color in color_groups:
        color = strip_accents(color).lower().strip()
        if color in {"white_sparkling", "blanco_espumante", "white+sparkling"}:
            expanded.extend(["white", "sparkling"])
        elif color in {"blanco", "white"}:
            expanded.append("white")
        elif color in {"tinto", "red"}:
            expanded.append("red")
        elif color in {"espumante", "sparkling"}:
            expanded.append("sparkling")
        elif color in {"rosado", "rose", "rose"}:
            expanded.append("rose")
        else:
            expanded.append(color)
    return set(expanded)


def term_usage_by_variety(term: str, color_groups=None, min_reviews: int = MIN_REVIEWS_PER_VARIETY) -> pd.DataFrame:
    term = canonical_term(term)
    color_groups = normalize_color_filter(color_groups)
    rows = []
    meta_lookup = variety_meta.set_index("variety")
    for variety, counter in variety_counters.items():
        if variety not in meta_lookup.index:
            continue
        meta = meta_lookup.loc[variety]
        if meta["review_count"] < min_reviews:
            continue
        if color_groups is not None and meta["color_group"] not in color_groups:
            continue
        total = sum(counter.values())
        count = counter.get(term, 0)
        review_hits = review_presence_counters.get(variety, Counter()).get(term, 0)
        rows.append({
            "variety": variety,
            "color_group": meta["color_group"],
            "review_count": int(meta["review_count"]),
            "term": term,
            "count": int(count),
            "per_1000_tokens": float(1000 * count / total) if total else 0.0,
            "review_hits": int(review_hits),
            "review_pct": float(100 * review_hits / meta["review_count"]) if meta["review_count"] else 0.0,
        })
    return pd.DataFrame(rows)
def plot_term_usage_by_variety(term: str, color_groups=None, top_n: int = 20, mode: str = "review_pct"):
    metric, label = TERM_USAGE_MODES.get(mode, TERM_USAGE_MODES["review_pct"])
    table = term_usage_by_variety(term, color_groups=color_groups).sort_values(metric, ascending=False).head(top_n)
    if table.empty:
        print(f"No hay datos para el termino: {term}")
        return table
    plot_table = table.sort_values(metric)
    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(plot_table))))
    ax.barh(plot_table["variety"], plot_table[metric])
    color_label = "todos" if color_groups is None else ", ".join(sorted(normalize_color_filter(color_groups)))
    ax.set_title(f"Cepas donde mas aparece '{canonical_term(term)}' ({color_label})")
    ax.set_xlabel(label)
    ax.set_ylabel("")
    plt.show()
    return table


plot_top_terms(["Cabernet Sauvignon", "Pinot Noir", "Chardonnay"], n=18, mode="review_pct")

# Nuevo: donde se repite mas una palabra del diccionario.
plot_term_usage_by_variety("cherry", top_n=20, mode="count")
plot_term_usage_by_variety("cherry", top_n=20, mode="review_pct", color_groups=["red"])
plot_term_usage_by_variety("fig", top_n=20, mode="review_pct", color_groups=["white", "sparkling"])

## Ejercicio 2: graficos radiales



El grafico radial muestra varios descriptores alrededor de un circulo. Cada eje es una palabra y cada linea representa una cepa.

Cuando una linea se aleja del centro en un eje, esa cepa usa mas ese descriptor. Cuando dos lineas tienen formas parecidas, sus perfiles de lenguaje son parecidos.

El radar inverso usa una palabra como centro conceptual y pone cepas en los ejes. Esa version ayuda a ver en que vinos una palabra, como fig o cherry, tiene mayor peso.


In [ ]:
def normalize_term(term: str) -> str:
    return strip_accents(term).lower().strip().replace(" ", "_").replace("-", "_")


def terms_for_color_group(color_group: str, top_n: int = 15) -> list[str]:
    varieties = variety_meta.loc[variety_meta["color_group"].eq(color_group), "variety"]
    counter = Counter()
    for variety in varieties:
        counter.update(variety_counters.get(variety, Counter()))
    return [term for term, _ in counter.most_common(top_n)]


def terms_for_varieties(varieties: list[str], top_n: int = 15) -> list[str]:
    counter = Counter()
    for variety in varieties:
        counter.update(variety_counters.get(variety, Counter()))
    return [term for term, _ in counter.most_common(top_n)]


def term_metric_value(variety: str, term: str, mode: str = "per_1000_tokens") -> float:
    term = canonical_term(term)
    counter = variety_counters[variety]
    total = sum(counter.values())
    count = counter.get(term, 0)
    if mode == "count":
        return float(count)
    if mode == "review_pct":
        meta = variety_meta.set_index("variety").loc[variety]
        hits = review_presence_counters.get(variety, Counter()).get(term, 0)
        return float(100 * hits / meta["review_count"]) if meta["review_count"] else 0.0
    return float(1000 * count / total) if total else 0.0


def radar_values(variety: str, terms: list[str], mode: str = "per_1000_tokens") -> np.ndarray:
    counter = variety_counters[variety]
    return np.array([term_metric_value(variety, term, mode=mode) for term in terms], dtype=float)


def top_varieties_for_terms(terms, color_groups=None, top_n: int = 12, mode: str = "review_pct") -> list[str]:
    terms = [canonical_term(term) for term in terms]
    color_groups = normalize_color_filter(color_groups)
    meta_lookup = variety_meta.set_index("variety")
    scored = []
    for variety, counter in variety_counters.items():
        if variety not in meta_lookup.index:
            continue
        color = meta_lookup.loc[variety, "color_group"]
        if color_groups is not None and color not in color_groups:
            continue
        total = sum(counter.values())
        if total == 0:
            continue
        score = sum(term_metric_value(variety, term, mode=mode) for term in terms)
        if score > 0:
            scored.append((variety, score))
    return [variety for variety, _ in sorted(scored, key=lambda item: item[1], reverse=True)[:top_n]]


def plot_radar_comparison(
    varieties: list[str],
    terms=None,
    top_n: int = 15,
    color_group=None,
    title=None,
    mode: str = "per_1000_tokens",
):
    if terms is None:
        if color_group is not None:
            terms = terms_for_color_group(color_group, top_n=top_n)
        else:
            terms = terms_for_varieties(varieties, top_n=top_n)
    terms = [canonical_term(term) for term in terms]

    angles = np.linspace(0, 2 * np.pi, len(terms), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={"polar": True})
    for variety in varieties:
        values = radar_values(variety, terms, mode=mode).tolist()
        values += values[:1]
        ax.plot(angles, values, linewidth=2, marker="o", label=variety)
        ax.fill(angles, values, alpha=0.08)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(terms, fontsize=9)
    ax.set_title(title or "Comparacion radial de palabras por cepa", y=1.08)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.show()

    return pd.DataFrame({variety: radar_values(variety, terms, mode=mode) for variety in varieties}, index=terms)


def plot_radar_term_pool(terms, color_groups, top_n_varieties: int = 12, title=None, mode: str = "review_pct"):
    terms = [canonical_term(term) for term in terms]
    varieties = top_varieties_for_terms(terms, color_groups=color_groups, top_n=top_n_varieties, mode=mode)
    if not varieties:
        print(f"No hay cepas con apariciones para: {terms}")
        return pd.DataFrame(index=varieties)

    angles = np.linspace(0, 2 * np.pi, len(varieties), endpoint=False).tolist()
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={"polar": True})
    values_by_term = {}

    for term in terms:
        values = [radar_values(variety, [term], mode=mode)[0] for variety in varieties]
        values_by_term[term] = values
        closed = values + values[:1]
        ax.plot(angles, closed, linewidth=2, marker="o", label=term)
        ax.fill(angles, closed, alpha=0.08)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(varieties, fontsize=8)
    ax.set_title(title or f"Palabras como centro: {', '.join(terms)}", y=1.08)
    ax.legend(loc="upper right", bbox_to_anchor=(1.28, 1.1))
    plt.show()

    return pd.DataFrame(values_by_term, index=varieties)


def plot_white_red_term_radars(terms, top_n_varieties: int = 12, mode: str = "review_pct"):
    terms = [canonical_term(term) for term in terms]
    white_like = ["white", "sparkling"]
    white_table = plot_radar_term_pool(
        terms,
        color_groups=white_like,
        top_n_varieties=top_n_varieties,
        mode=mode,
        title=f"Blancos y espumantes: {', '.join(terms)}",
    )
    red_table = plot_radar_term_pool(
        terms,
        color_groups=["red"],
        top_n_varieties=top_n_varieties,
        mode=mode,
        title=f"Tintos: {', '.join(terms)}",
    )
    return white_table, red_table


red_terms = terms_for_color_group("red", top_n=15)
white_terms = terms_for_color_group("white", top_n=15)
print("Red terms:", red_terms)
print("White terms:", white_terms)

## Ejemplo radial con vinos tintos



Este ejemplo compara Cabernet Sauvignon, Merlot y Malbec usando descriptores frecuentes de vinos tintos. La forma de cada linea resume si una cepa se inclina mas hacia fruta negra, fruta roja, especias, madera u otros descriptores.

La lectura no dice que una cepa sea mejor que otra. Solo muestra diferencias en el lenguaje de las reviews.


In [ ]:
plot_radar_comparison(
    ["Cabernet Sauvignon", "Merlot", "Malbec"],
    color_group="red",
    top_n=15,
    title="Tintos: Cabernet Sauvignon vs Merlot vs Malbec",
)

## Ejemplo radial con vinos blancos



Este ejemplo compara Chardonnay, Sauvignon Blanc y Riesling. En blancos suele ser util separar notas citricas, minerales, frutales y de madera.

Si una linea crece en citrus, lemon o grapefruit, significa que esas palabras aparecen con mayor frecuencia relativa en las reviews de esa cepa.


In [ ]:
plot_radar_comparison(
    ["Chardonnay", "Sauvignon Blanc", "Riesling"],
    color_group="white",
    top_n=15,
    title="Blancos: Chardonnay vs Sauvignon Blanc vs Riesling",
)

## Radiales con terminos elegidos



Aqui se prueban palabras seleccionadas desde el diccionario generado. El usuario no inventa texto libre, sino que elige conceptos ya detectados por el analisis.

Esto mantiene la comparacion controlada y evita que una palabra escrita de forma distinta genere una busqueda vacia o enga?osa.


In [ ]:
custom_terms = ["cocoa", "chocolate", "black_cherry", "plum", "vanilla", "spice", "oak", "pepper"]
plot_radar_comparison(
    ["Cabernet Sauvignon", "Malbec"],
    terms=custom_terms,
    title="Cabernet Sauvignon vs Malbec: terminos elegidos",
)

# Nuevo: radar inverso por pool de palabras. Muestra las cepas donde mas se repiten esas palabras.
plot_white_red_term_radars(["cherry"], top_n_varieties=10)
plot_white_red_term_radars(["fig"], top_n_varieties=10)

## Ejercicio 3: embedding con TF-IDF y SVD



El embedding convierte cada cepa en un perfil numerico basado en sus descriptores. TF-IDF destaca palabras caracteristicas y SVD reduce la matriz a dos dimensiones visibles.

Esta alternativa reemplaza TensorFlow para mantener compatibilidad con Python 3.14. El resultado es estable, rapido y suficiente para crear mapas exploratorios.

La posicion en el mapa no es geografia. Es una representacion visual de similitud de lenguaje.


In [ ]:
EMBEDDING_MIN_REVIEWS = 200
MAX_EMBEDDING_TERMS = 3000
SVD_COMPONENTS = 2


def build_variety_documents(min_reviews: int = EMBEDDING_MIN_REVIEWS) -> pd.DataFrame:
    rows = []
    meta_lookup = variety_meta.set_index("variety")
    for variety, counter in variety_counters.items():
        if variety not in meta_lookup.index:
            continue
        meta = meta_lookup.loc[variety]
        if meta["review_count"] < min_reviews:
            continue
        tokens = []
        for term, count in counter.items():
            tokens.extend([term] * int(count))
        if not tokens:
            continue
        rows.append({
            "variety": variety,
            "color_group": meta["color_group"],
            "review_count": int(meta["review_count"]),
            "document": " ".join(tokens),
        })
    return pd.DataFrame(rows).sort_values("review_count", ascending=False).reset_index(drop=True)


embedding_docs = build_variety_documents()
tfidf_vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    max_features=MAX_EMBEDDING_TERMS,
)
tfidf_matrix = tfidf_vectorizer.fit_transform(embedding_docs["document"])
svd_model = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=RANDOM_SEED)
raw_variety_embedding_matrix = svd_model.fit_transform(tfidf_matrix)
embedding_terms = np.array(tfidf_vectorizer.get_feature_names_out())
raw_word_embedding_matrix = svd_model.components_.T

# Centro visual: el 0,0 es el centroide imaginario de las palabras descriptoras.
# Escalamos por dispersion de palabras para que el mapa sea legible al ojo humano.
word_centroid = raw_word_embedding_matrix.mean(axis=0, keepdims=True)
word_scale = raw_word_embedding_matrix.std(axis=0, keepdims=True)
word_scale = np.where(word_scale == 0, 1, word_scale)
variety_embedding_matrix = (raw_variety_embedding_matrix - word_centroid) / word_scale
word_embedding_matrix = (raw_word_embedding_matrix - word_centroid) / word_scale
vocab_index = {term: idx for idx, term in enumerate(embedding_terms)}
variety_index = {variety: idx for idx, variety in enumerate(embedding_docs["variety"])}
embedding_coords = pd.DataFrame({
    "label": embedding_docs["variety"],
    "kind": "variety",
    "color_group": embedding_docs["color_group"],
    "x": variety_embedding_matrix[:, 0],
    "y": variety_embedding_matrix[:, 1],
})

display(embedding_docs[["variety", "color_group", "review_count"]].head(20))
print("Variedades en embedding:", len(embedding_docs))
print("Terminos en embedding:", len(embedding_terms))
print("Varianza explicada SVD:", svd_model.explained_variance_ratio_.round(4).tolist())

## Varianza explicada del mapa reducido



Este grafico de barras muestra cuanta informacion conserva cada eje del mapa SVD. Una barra mas alta indica que ese eje resume mas diferencias del vocabulario.

Para un lector no tecnico, esta visualizacion sirve como indicador de cuanto peso tiene cada dimension del mapa.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(["SVD 1", "SVD 2"], svd_model.explained_variance_ratio_)
ax.set_title("Varianza explicada por componentes SVD")
ax.set_ylabel("Proporcion")
plt.show()

## Muestra de coordenadas



Esta tabla muestra algunas coordenadas del embedding. Los numeros son posiciones matematicas usadas para dibujar el mapa.

No se interpretan de forma aislada; su valor aparece al comparar distancias entre cepas y palabras.


In [ ]:
embedding_coords.head()

## Busquedas de cercania



Estas funciones conectan cepas con sus palabras cercanas y palabras con sus cepas cercanas. La cercania indica parecido en el uso de descriptores.

Cuando se buscan dos palabras, se resaltan cepas comunes a ambas. Esto permite encontrar variedades que comparten varios rasgos sensoriales al mismo tiempo.


In [ ]:
def word_vector(term: str):
    idx = vocab_index.get(canonical_term(term))
    if idx is None:
        return None
    return word_embedding_matrix[idx]


def nearest_words_to_variety(variety: str, n: int = 12, mode: str = "review_pct") -> pd.DataFrame:
    if variety not in variety_index:
        return pd.DataFrame(columns=["term", "score"])
    counter = variety_counters.get(variety, Counter())
    rows = []
    for term in counter:
        if term in vocab_index:
            rows.append({"term": term, "score": term_metric_value(variety, term, mode=mode)})
    return pd.DataFrame(rows).sort_values("score", ascending=False).head(n).reset_index(drop=True)


def nearest_varieties_to_word(term: str, n: int = 3, mode: str = "review_pct") -> pd.DataFrame:
    term = canonical_term(term)
    if term not in vocab_index and term not in total_counter:
        return pd.DataFrame(columns=["variety", "color_group", "score"])
    table = term_usage_by_variety(term).sort_values(TERM_USAGE_MODES.get(mode, TERM_USAGE_MODES["review_pct"])[0], ascending=False).head(n)
    metric = TERM_USAGE_MODES.get(mode, TERM_USAGE_MODES["review_pct"])[0]
    return table[["variety", "color_group", metric]].rename(columns={metric: "score"}).reset_index(drop=True)


def common_varieties_for_words(words, n_per_word: int = 3, mode: str = "review_pct") -> pd.DataFrame:
    neighbor_tables = [nearest_varieties_to_word(word, n=n_per_word, mode=mode) for word in words]
    if not neighbor_tables:
        return pd.DataFrame(columns=["variety", "hits", "mean_score"])
    all_neighbors = pd.concat(
        [table.assign(query_word=canonical_term(word)) for word, table in zip(words, neighbor_tables)],
        ignore_index=True,
    )
    if all_neighbors.empty:
        return pd.DataFrame(columns=["variety", "hits", "mean_score"])
    return (
        all_neighbors.groupby("variety", as_index=False)
        .agg(hits=("query_word", "nunique"), mean_score=("score", "mean"))
        .sort_values(["hits", "mean_score"], ascending=False)
        .reset_index(drop=True)
    )

## Mapa local de cepas y palabras



El mapa local se centra en la comparacion seleccionada. Con una cepa, esa cepa queda en el centro. Con dos cepas, el punto medio representa lo compartido. Con tres o mas, las cepas parten como una figura regular y luego se acercan o separan por similitud real.

Una palabra compartida cae cerca del centro de las cepas que la comparten. Una palabra casi exclusiva queda detras de la cepa donde domina, para no insinuar que tambien pertenece a las otras.

El mapa grande dibuja todo el vocabulario registrado como puntos. Solo se etiquetan las palabras mas relevantes para que una persona pueda leerlo sin perderse en miles de etiquetas.


In [ ]:
def stable_angle(label: str) -> float:
    digest = hashlib.md5(str(label).encode("utf-8")).hexdigest()
    return (int(digest[:8], 16) / 16**8) * 2 * np.pi


def variety_similarity(left: str, right: str) -> float:
    return float(cosine_similarity(tfidf_matrix[variety_index[left]], tfidf_matrix[variety_index[right]])[0, 0])


def regular_polygon_positions(labels, radius: float = 1.45):
    start_angle = np.pi / 2
    if len(labels) == 4:
        start_angle = np.pi / 4
    return {
        label: np.array([
            np.cos(start_angle + 2 * np.pi * i / len(labels)),
            np.sin(start_angle + 2 * np.pi * i / len(labels)),
        ]) * radius
        for i, label in enumerate(labels)
    }


def selected_variety_positions(varieties, iterations: int = 160):
    varieties = [variety for variety in varieties if variety in variety_index]
    if len(varieties) == 1:
        return {varieties[0]: np.array([0.0, 0.0])}
    if len(varieties) == 2:
        left, right = varieties
        sim = variety_similarity(left, right)
        half_distance = max(0.75, min(2.2, 0.65 + (1 - sim) * 2.5))
        return {left: np.array([-half_distance, 0.0]), right: np.array([half_distance, 0.0])}

    positions = regular_polygon_positions(varieties, radius=1.55 if len(varieties) <= 4 else 1.9)
    similarities = {}
    distances = []
    for i, left in enumerate(varieties):
        for right in varieties[i + 1:]:
            sim = variety_similarity(left, right)
            similarities[(left, right)] = sim
            distances.append(1 - sim)
    min_distance = min(distances) if distances else 0
    max_distance = max(distances) if distances else 1
    span = max(max_distance - min_distance, 1e-9)

    for _ in range(iterations):
        for (left, right), sim in similarities.items():
            raw_distance = 1 - sim
            normalized_distance = (raw_distance - min_distance) / span
            target = 0.85 + normalized_distance * 2.35
            delta = positions[right] - positions[left]
            current = np.linalg.norm(delta)
            if current == 0:
                angle = stable_angle(left + right)
                delta = np.array([np.cos(angle), np.sin(angle)]) * 1e-3
                current = np.linalg.norm(delta)
            unit = delta / current
            correction = (current - target) * 0.045 * unit
            positions[left] += correction
            positions[right] -= correction
        centroid = np.mean(np.vstack(list(positions.values())), axis=0)
        for variety in positions:
            positions[variety] -= centroid

    max_radius = max(np.linalg.norm(position) for position in positions.values())
    if max_radius > 0:
        scale = (1.75 if len(varieties) <= 4 else 2.2) / max_radius
        for variety in positions:
            positions[variety] *= scale
    return positions


def local_word_position(term, variety_positions, max_scores):
    weights = []
    positions = []
    for variety, position in variety_positions.items():
        score = term_metric_value(variety, term, mode="review_pct")
        weights.append(score)
        positions.append(position)
    weights = np.array(weights, dtype=float)
    positions = np.vstack(positions)
    if weights.sum() == 0:
        angle = stable_angle(term)
        return np.array([np.cos(angle), np.sin(angle)]) * 3.0
    base = np.average(positions, axis=0, weights=weights)
    strength = weights.max() / max(max_scores, 1e-9)
    if len(variety_positions) == 1:
        angle = stable_angle(term)
        distance = 0.25 + (1 - strength) * 2.5
        return np.array([np.cos(angle), np.sin(angle)]) * distance
    if len(variety_positions) == 2:
        p1, p2 = positions[0], positions[1]
        axis = p2 - p1
        axis_len = np.linalg.norm(axis)
        if axis_len == 0:
            axis = np.array([1.0, 0.0])
            axis_len = 1.0
        unit_axis = axis / axis_len
        midpoint = (p1 + p2) / 2
        half_distance = axis_len / 2
        signed_balance = (weights[1] - weights[0]) / max(weights.max(), 1e-9)
        signed_balance = float(np.clip(signed_balance, -1.0, 1.0))
        dominance = abs(signed_balance)
        beyond = max(0.0, dominance - 0.70) / 0.30 * half_distance * 0.75
        line_distance = dominance * half_distance + beyond
        direction = 1 if signed_balance >= 0 else -1
        base = midpoint + unit_axis * direction * line_distance
        perp = np.array([-unit_axis[1], unit_axis[0]])
        jitter_sign = 1 if math.sin(stable_angle(term)) >= 0 else -1
        jitter = perp * jitter_sign * (0.06 + (1 - strength) * 0.22)
        return base + jitter
    jitter_angle = stable_angle(term)
    jitter = np.array([np.cos(jitter_angle), np.sin(jitter_angle)]) * (0.05 + (1 - strength) * 0.35)
    return base + jitter


def plot_embedding_search(
    varieties=None,
    words=None,
    nearest_word_count: int = 14,
    nearest_variety_count: int = 3,
    include_all_words: bool = False,
    max_all_words=None,
    label_limit=None,
    figsize=(11, 8),
    dpi: int = 110,
    save_path=None,
):
    varieties = [variety for variety in (varieties or []) if variety in variety_index]
    words = [canonical_term(word) for word in (words or [])]
    rows = []
    word_neighbor_tables = []

    if varieties:
        variety_positions = selected_variety_positions(varieties)
        for variety, position in variety_positions.items():
            rows.append({"label": variety, "kind": "query_variety", "x": position[0], "y": position[1], "highlighted": True, "score": np.inf})

        candidate_terms = set(words)
        if include_all_words:
            if max_all_words is None:
                candidate_terms.update(embedding_terms)
            else:
                candidate_terms.update(term for term, _ in total_counter.most_common(max_all_words) if term in vocab_index)
        else:
            for variety in varieties:
                candidate_terms.update(nearest_words_to_variety(variety, n=nearest_word_count, mode="review_pct")["term"])
        max_scores = max(
            [term_metric_value(variety, term, mode="review_pct") for variety in varieties for term in candidate_terms] or [1]
        )
        for term in sorted(candidate_terms):
            if term not in vocab_index and term not in total_counter:
                continue
            score = max(term_metric_value(variety, term, mode="review_pct") for variety in varieties)
            position = local_word_position(term, variety_positions, max_scores)
            rows.append({"label": term, "kind": "query_word" if term in words else "near_word", "x": position[0], "y": position[1], "highlighted": term in words, "score": score})
    else:
        valid_words = [word for word in words if word in vocab_index or word in total_counter]
        if len(valid_words) == 1:
            word_positions = {valid_words[0]: np.array([0.0, 0.0])}
        else:
            word_positions = {}
            for i, word in enumerate(valid_words):
                angle = 2 * np.pi * i / max(len(valid_words), 1)
                word_positions[word] = np.array([np.cos(angle), np.sin(angle)]) * 0.7
        for word, position in word_positions.items():
            rows.append({"label": word, "kind": "query_word", "x": position[0], "y": position[1], "highlighted": True, "score": np.inf})
            neighbors = nearest_varieties_to_word(word, n=nearest_variety_count, mode="review_pct")
            word_neighbor_tables.append(neighbors.assign(query_word=word))
        common_table = common_varieties_for_words(valid_words, n_per_word=nearest_variety_count, mode="review_pct") if len(valid_words) > 1 else pd.DataFrame()
        all_neighbor_varieties = pd.concat(word_neighbor_tables, ignore_index=True)["variety"].unique() if word_neighbor_tables else []
        for variety in all_neighbor_varieties:
            weights = np.array([term_metric_value(variety, word, mode="review_pct") for word in valid_words], dtype=float)
            positions = np.vstack([word_positions[word] for word in valid_words]) if valid_words else np.zeros((1, 2))
            if weights.sum() > 0 and len(valid_words) > 1:
                position = np.average(positions, axis=0, weights=weights)
            else:
                angle = stable_angle(variety)
                distance = 0.6 + (1 - min(weights.max() / max(weights.max(), 1e-9), 1)) * 1.5
                position = list(word_positions.values())[0] + np.array([np.cos(angle), np.sin(angle)]) * distance
            is_common = not common_table.empty and variety in set(common_table.loc[common_table["hits"].gt(1), "variety"])
            rows.append({"label": variety, "kind": "common_variety" if is_common else "near_variety", "x": position[0], "y": position[1], "highlighted": is_common, "score": float(weights.max()) if len(weights) else 0.0})

    if not rows:
        print("No hay elementos validos para graficar.")
        return pd.DataFrame(columns=["label", "kind", "x", "y"])

    plot_df = pd.DataFrame(rows).drop_duplicates(subset=["label", "kind"]).reset_index(drop=True)
    if label_limit is None:
        label_limit = 90 if include_all_words else len(plot_df)
    always_label = set(plot_df.loc[plot_df["highlighted"] | plot_df["kind"].isin(["query_variety", "query_word"]), "label"])
    scored_labels = set(
        plot_df.loc[~plot_df["label"].isin(always_label)]
        .sort_values("score", ascending=False)
        .head(max(0, label_limit - len(always_label)))["label"]
    )
    label_set = always_label | scored_labels

    near_word_size = 14 if include_all_words else 55
    near_word_alpha = 0.35 if include_all_words else 0.85
    styles = {
        "query_variety": ("s", 170),
        "query_word": ("*", 210),
        "near_word": ("o", near_word_size),
        "near_variety": ("^", 95),
        "common_variety": ("D", 150),
    }
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    for kind, group in plot_df.groupby("kind"):
        marker, size = styles.get(kind, ("o", 50))
        alpha = near_word_alpha if kind == "near_word" else 0.85
        ax.scatter(group["x"], group["y"], s=size, marker=marker, label=kind, alpha=alpha)
    for _, row in plot_df.iterrows():
        if row["label"] not in label_set:
            continue
        weight = "bold" if row["highlighted"] else "normal"
        ax.text(row["x"], row["y"], "  " + row["label"], fontsize=9, weight=weight, va="center")
    ax.axhline(0, color="black", linewidth=0.8, alpha=0.25)
    ax.axvline(0, color="black", linewidth=0.8, alpha=0.25)
    ax.set_title("Mapa local: cepas y descriptores")
    ax.set_xlabel("Eje local 1")
    ax.set_ylabel("Eje local 2")
    ax.set_aspect("equal", adjustable="datalim")
    ax.legend()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches="tight", dpi=dpi)
    plt.show()
    if word_neighbor_tables:
        display(pd.concat(word_neighbor_tables, ignore_index=True))
    return plot_df


def plot_full_local_word_map(varieties, words=None, max_words=None, label_top_n: int = 120, figsize=(18, 14), dpi: int = 180, save_path=None):
    return plot_embedding_search(
        varieties=varieties,
        words=words,
        nearest_word_count=0,
        include_all_words=True,
        max_all_words=max_words,
        label_limit=label_top_n,
        figsize=figsize,
        dpi=dpi,
        save_path=save_path,
    )


plot_embedding_search(varieties=["Cabernet Sauvignon", "Malbec"], words=["cocoa", "chocolate"])
plot_embedding_search(varieties=["Cabernet Sauvignon", "Malbec", "Chardonnay"], nearest_word_count=12, figsize=(10, 9), dpi=130)
plot_embedding_search(varieties=["Cabernet Sauvignon", "Malbec", "Pinot Noir", "Syrah"], nearest_word_count=12, figsize=(10, 10), dpi=130)
plot_full_local_word_map(
    ["Cabernet Sauvignon", "Malbec", "Pinot Noir", "Chardonnay"],
    label_top_n=120,
    save_path=DATA_DIR / "app" / "figures" / "full_local_word_map.png",
)


## Export para la futura app web



Esta etapa genera un JSON liviano con diccionario, cepas, descriptores y coordenadas. La futura app HTML, CSS y JavaScript podra leer ese archivo sin cargar los CSV completos.

La intencion final es transformar este analisis en una interfaz donde se puedan seleccionar vinos, comparar palabras, ver radiales y navegar el mapa local de la matriz de vinos.

El proyecto queda listo como portafolio y como base tecnica para esa app.


In [ ]:
def export_app_payload(max_terms_per_variety: int = 250, max_dictionary_terms: int = 3000):
    export_dir = DATA_DIR / "app" / "data"
    export_dir.mkdir(parents=True, exist_ok=True)

    meta_by_variety = variety_meta.set_index("variety").to_dict(orient="index")
    varieties_payload = []
    for variety, counter in variety_counters.items():
        total = sum(counter.values())
        review_count = int(meta_by_variety.get(variety, {}).get("review_count", 0))
        top_terms = []
        for term, count in counter.most_common(max_terms_per_variety):
            review_hits = review_presence_counters.get(variety, Counter()).get(term, 0)
            top_terms.append({
                "term": term,
                "count": int(count),
                "per_1000_tokens": float(1000 * count / total),
                "review_hits": int(review_hits),
                "review_pct": float(100 * review_hits / review_count) if review_count else 0.0,
            })
        info = meta_by_variety.get(variety, {})
        varieties_payload.append({
            "variety": variety,
            "color_group": info.get("color_group", "unknown"),
            "review_count": int(info.get("review_count", 0)),
            "token_count": int(total),
            "top_terms": top_terms,
        })

    dictionary = [
        {"term": term, "count": int(count)}
        for term, count in total_counter.most_common(max_dictionary_terms)
    ]

    payload = {
        "meta": {
            "min_reviews_per_variety": MIN_REVIEWS_PER_VARIETY,
            "max_terms_per_variety": max_terms_per_variety,
            "max_dictionary_terms": max_dictionary_terms,
            "quality_filter_enabled": False,
        },
        "dictionary": dictionary,
        "varieties": varieties_payload,
    }

    if "word_embedding_matrix" in globals() and "vocab_index" in globals() and len(vocab_index) > 0:
        selected_terms = [row["term"] for row in dictionary if row["term"] in vocab_index]
        selected_varieties = [row["variety"] for row in varieties_payload if row["variety"] in variety_counters]

        labels = []
        kinds = []
        vectors = []

        for variety in selected_varieties:
            if variety not in variety_index:
                continue
            labels.append(variety)
            kinds.append("variety")
            vectors.append(variety_embedding_matrix[variety_index[variety]])
        for term in selected_terms:
            labels.append(term)
            kinds.append("word")
            vectors.append(word_embedding_matrix[vocab_index[term]])

        if vectors:
            payload["embedding_2d"] = [
                {"label": label, "kind": kind, "x": float(x), "y": float(y)}
                for label, kind, (x, y) in zip(labels, kinds, np.vstack(vectors))
            ]

    out_path = export_dir / "wine_variety_language_profiles.json"
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    return out_path


export_path = export_app_payload()
export_path